In [1]:
!pip -q install sentence-transformers faiss-cpu tqdm

In [2]:
from pathlib import Path
import json
import os
from collections import Counter

import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

torch: 2.11.0+cu128
cuda available: True


In [3]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/CareGuide Agent')
CHUNKS_PATH = PROJECT_DIR / 'data' / 'processed' / 'official_rag_chunks.jsonl'
INDEX_DIR = PROJECT_DIR / 'data' / 'indexes'

print('CHUNKS_PATH:', CHUNKS_PATH)
print('exists:', CHUNKS_PATH.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CHUNKS_PATH: /content/drive/MyDrive/CareGuide Agent/data/processed/official_rag_chunks.jsonl
exists: True


In [4]:
from pathlib import Path

matches = list(Path('/content/drive/MyDrive').rglob('official_rag_chunks.jsonl'))

print('matches:', len(matches))
for path in matches:
    print(path)

matches: 1
/content/drive/MyDrive/CareGuide Agent/data/processed/official_rag_chunks.jsonl


In [5]:
MODEL_NAME = 'intfloat/multilingual-e5-base'
BATCH_SIZE = 64

INDEX_DIR.mkdir(parents=True, exist_ok=True)
INDEX_PATH = INDEX_DIR / 'official_rag_faiss.index'
METADATA_PATH = INDEX_DIR / 'official_rag_chunk_metadata.jsonl'
MANIFEST_PATH = INDEX_DIR / 'official_rag_index_manifest.json'

print('INDEX_DIR:', INDEX_DIR)

INDEX_DIR: /content/drive/MyDrive/CareGuide Agent/data/indexes


In [6]:
REQUIRED_FIELDS = {
    'chunk_id',
    'document_id',
    'source',
    'url',
    'title',
    'topic',
    'section_heading',
    'section_type',
    'text',
}

def load_chunks(path: Path) -> list[dict]:
    chunks = []
    with path.open('r', encoding='utf-8') as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()
            if not line:
                continue
            item = json.loads(line)
            missing = REQUIRED_FIELDS - set(item)
            if missing:
                raise ValueError(f'{path}:{line_number}: missing fields {sorted(missing)}')
            if not item['text'].strip():
                raise ValueError(f'{path}:{line_number}: empty text')
            chunks.append(item)
    return chunks

chunks = load_chunks(CHUNKS_PATH)
print('chunks:', len(chunks))
print('source counts:', dict(Counter(chunk['source'] for chunk in chunks)))
print('first chunk:', {key: chunks[0][key] for key in ['chunk_id', 'source', 'title', 'section_type', 'word_count'] if key in chunks[0]})

chunks: 10451
source counts: {'NHS': 6319, 'CDC': 145, 'MedlinePlus': 3987}
first chunk: {'chunk_id': 'nhs_chest_pain__section_0__chunk_0', 'source': 'NHS', 'title': 'Chest pain', 'section_type': 'immediate_action', 'word_count': 86}


In [7]:
def passage_text(chunk: dict) -> str:
    parts = [
        str(chunk.get('title', '')),
        str(chunk.get('section_heading', '')),
        str(chunk.get('text', '')),
    ]
    text = '\n'.join(part.strip() for part in parts if part and part.strip())
    return f'passage: {text}'

passages = [passage_text(chunk) for chunk in chunks]
print(passages[0][:500])

passage: Chest pain
Call 999 if:
Call 999 if: - you get sudden pain or discomfort in your chest that does not go away – the pain can feel like squeezing or pressure inside your chest, burning or indigestion - you get pain that spreads to your left or right arm, or your neck, jaw, stomach or back - you have chest pain and you feel sweaty, sick, light headed or short of breath You could be having a heart attack. Call 999 straight away as you need immediate treatment in hospital.


In [8]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer(MODEL_NAME, device=device)
print('model:', MODEL_NAME)
print('device:', device)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

model: intfloat/multilingual-e5-base
device: cuda


In [9]:
embedding_batches = []

for start in tqdm(range(0, len(passages), BATCH_SIZE)):
    batch = passages[start:start + BATCH_SIZE]
    batch_embeddings = model.encode(
        batch,
        batch_size=BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    embedding_batches.append(batch_embeddings.astype('float32'))

embeddings = np.vstack(embedding_batches).astype('float32')
print('embeddings shape:', embeddings.shape)
print('dtype:', embeddings.dtype)

  0%|          | 0/164 [00:00<?, ?it/s]

embeddings shape: (10451, 768)
dtype: float32


In [10]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

faiss.write_index(index, str(INDEX_PATH))
print('saved index:', INDEX_PATH)
print('index vectors:', index.ntotal)

saved index: /content/drive/MyDrive/CareGuide Agent/data/indexes/official_rag_faiss.index
index vectors: 10451


In [11]:
METADATA_FIELDS = [
    'chunk_id',
    'document_id',
    'source',
    'url',
    'title',
    'topic',
    'priority',
    'symptom_group',
    'section_index',
    'section_heading',
    'section_type',
    'chunk_index',
    'word_count',
    'text',
]

with METADATA_PATH.open('w', encoding='utf-8') as file:
    for chunk in chunks:
        metadata = {field: chunk.get(field) for field in METADATA_FIELDS if field in chunk}
        file.write(json.dumps(metadata, ensure_ascii=False) + '\n')

manifest = {
    'model_name': MODEL_NAME,
    'embedding_dimension': int(dimension),
    'normalize_embeddings': True,
    'faiss_metric': 'inner_product_cosine_with_normalized_vectors',
    'chunk_count': len(chunks),
    'source_counts': dict(Counter(chunk['source'] for chunk in chunks)),
    'chunks_path': str(CHUNKS_PATH),
    'index_path': str(INDEX_PATH),
    'metadata_path': str(METADATA_PATH),
}

MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

print('saved metadata:', METADATA_PATH)
print('saved manifest:', MANIFEST_PATH)
print(json.dumps(manifest, ensure_ascii=False, indent=2))

saved metadata: /content/drive/MyDrive/CareGuide Agent/data/indexes/official_rag_chunk_metadata.jsonl
saved manifest: /content/drive/MyDrive/CareGuide Agent/data/indexes/official_rag_index_manifest.json
{
  "model_name": "intfloat/multilingual-e5-base",
  "embedding_dimension": 768,
  "normalize_embeddings": true,
  "faiss_metric": "inner_product_cosine_with_normalized_vectors",
  "chunk_count": 10451,
  "source_counts": {
    "NHS": 6319,
    "CDC": 145,
    "MedlinePlus": 3987
  },
  "chunks_path": "/content/drive/MyDrive/CareGuide Agent/data/processed/official_rag_chunks.jsonl",
  "index_path": "/content/drive/MyDrive/CareGuide Agent/data/indexes/official_rag_faiss.index",
  "metadata_path": "/content/drive/MyDrive/CareGuide Agent/data/indexes/official_rag_chunk_metadata.jsonl"
}


## Test retrieval nhanh

E5 dùng prefix `query:` cho câu hỏi và `passage:` cho tài liệu. Cell dưới đây kiểm tra index có trả về chunk liên quan không.

In [12]:
def search(query: str, top_k: int = 5) -> list[dict]:
    query_embedding = model.encode(
        [f'query: {query}'],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype('float32')
    scores, indices = index.search(query_embedding, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        chunk = chunks[int(idx)]
        results.append({
            'score': float(score),
            'chunk_id': chunk['chunk_id'],
            'source': chunk['source'],
            'title': chunk['title'],
            'section_type': chunk['section_type'],
            'url': chunk['url'],
            'preview': chunk['text'][:400],
        })
    return results

results = search('Tôi bị đau ngực và khó thở, khi nào cần đi cấp cứu?', top_k=5)
for item in results:
    print('\n---')
    print('score:', round(item['score'], 4))
    print('source:', item['source'])
    print('title:', item['title'])
    print('section_type:', item['section_type'])
    print('url:', item['url'])
    print(item['preview'])


---
score: 0.834
source: NHS
title: Antiphospholipid syndrome (APS)
section_type: immediate_action
url: https://www.nhs.uk/conditions/antiphospholipid-syndrome/
Call 999 or go to A&E now if: - you have severe difficulty breathing - you feel pain in your chest or upper back - your heart is beating very fast - someone has passed out These could be signs of a pulmonary embolism or another serious condition which could be life-threatening and needs treatment straight away.

---
score: 0.8318
source: NHS
title: Acute respiratory distress syndrome (ARDS)
section_type: immediate_action
url: https://www.nhs.uk/conditions/acute-respiratory-distress-syndrome/
Call 999 or go to A&E immediately if you or someone else: - have severe difficulty breathing, for example, not being able to get words out, choking, or gasping - have sudden shortness of breath and pain in your arms, back, neck or jaw - have sudden shortness of breath and your chest feels tight or heavy - have sudden shortness of breath an